# Mini-Capstone — the full applied project

**Educational purpose:** this notebook mirrors the Streamlit *Mini-Capstone* page for
someone who wants to read the **code**, not click an app. We walk one product question
through the entire data-science workflow:

> *Which users retain, monetize, or churn — and what should the team do about it?*

Steps: inspect the relational data → engineer user-level features → EDA → a D7-retention
classifier → a 30-day revenue regressor → a K-means + PCA segmentation → a written PM memo.
Everything runs on a deterministic synthetic product-analytics dataset generated locally
into SQLite, so the notebook is fully self-contained.

## Setup

Put the project root on `sys.path` so `import src.*` works whether the notebook is run from
`notebooks/` or from the project root, then import the shared data-science layer.

In [ ]:
import sys, pathlib

# Robust project-root detection: try cwd and its parent, pick the one that
# actually contains the `src` package, so this works from notebooks/ OR root.
_here = pathlib.Path.cwd()
_candidates = [_here, _here.parent] + list(_here.parents)
PROJECT_ROOT = next((p for p in _candidates if (p / "src").is_dir()), _here.parent)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.db import ensure_database, table_overview, load_table, run_query
from src.feature_engineering import (
    build_modeling_frame,
    get_classification_xy,
    get_regression_xy,
    FEATURE_COLUMNS,
)
from src.modeling import (
    train_classifier,
    feature_importance,
    train_regressor,
    run_kmeans_pca,
    cluster_profile,
)
from src.metrics import classification_metrics, confusion_counts, regression_metrics

# Make sure the synthetic SQLite database exists before we read it.
ensure_database()
print("database ready")

## 1. Inspect the tables

The schema is relational: `users` (one row per user) plus many-rows-per-user `sessions`,
`events`, and `transactions`, and a `labels` table holding the outcomes we want to predict.

In [ ]:
# Row counts per table.
table_overview()

In [ ]:
# A peek at the users and labels tables.
display(load_table("users").head())
display(load_table("labels").head())

In [ ]:
# A real join: every user, their session count, and their labels (LEFT JOIN keeps
# users with no sessions / no purchases instead of silently dropping them).
sql = """
SELECT u.user_id, u.acquisition_channel,
       COUNT(s.session_id) AS n_sessions,
       l.retained_d7, l.total_revenue_30d
FROM users AS u
LEFT JOIN sessions AS s ON s.user_id = u.user_id
LEFT JOIN labels   AS l ON l.user_id = u.user_id
GROUP BY u.user_id, u.acquisition_channel, l.retained_d7, l.total_revenue_30d
ORDER BY n_sessions DESC
LIMIT 5;
"""
run_query(sql)

## 2. Feature engineering

Models never see raw logs. `build_modeling_frame()` aggregates the event/session/transaction
tables into **one row per user** (first-week behavior) and joins the labels. All features are
measured *before* the targets, which avoids leakage.

In [ ]:
frame = build_modeling_frame()
print("shape:", frame.shape)
print("features:", FEATURE_COLUMNS)
frame.head()

## 3. EDA

Look before you model. Two quick views: D7 retention by acquisition channel (a comparison),
and the relationship between early sessions and 30-day revenue.

In [ ]:
users = load_table("users")[["user_id", "acquisition_channel"]]
eda = frame.merge(users, on="user_id", how="left")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# (a) retention by channel
ret_by_channel = eda.groupby("acquisition_channel")["retained_d7"].mean().sort_values()
axes[0].bar(ret_by_channel.index, ret_by_channel.values, color="#7aa2f7")
axes[0].set_title("D7 retention by channel")
axes[0].tick_params(axis="x", rotation=30)

# (b) sessions vs revenue
axes[1].scatter(eda["total_sessions_7d"], eda["total_revenue_30d"], alpha=0.3, s=10, color="#9ece6a")
axes[1].set_xlabel("sessions (first 7d)")
axes[1].set_ylabel("revenue (30d)")
axes[1].set_title("Sessions vs revenue")

fig.tight_layout()
plt.show()

## 4. D7 retention classifier

Predict the binary `retained_d7` with a Random Forest, scored on a held-out split, and
inspect the confusion matrix plus feature importances.

In [ ]:
Xc, yc, _ = get_classification_xy("retained_d7")
clf_res = train_classifier(Xc, yc, "Random Forest")

clf_metrics = classification_metrics(clf_res.y_test, clf_res.y_pred, clf_res.y_proba)
print("D7 classifier metrics:")
for k, v in clf_metrics.items():
    print(f"  {k:10s} {v:.3f}")

imp = feature_importance(clf_res)
print("\nTop drivers of retention:")
print(imp.head(5).to_string(index=False))

In [ ]:
# Confusion matrix on the held-out set.
counts = confusion_counts(clf_res.y_test, clf_res.y_pred)
cm = np.array([[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]])

fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, fontweight="bold")
ax.set_xticks([0, 1], ["Pred churn", "Pred retain"])
ax.set_yticks([0, 1], ["True churn", "True retain"])
ax.set_title("Confusion matrix (D7 retention)")
fig.tight_layout()
plt.show()

## 5. 30-day revenue regressor

A numeric target: `total_revenue_30d`. Same workflow, regression metrics. Revenue is
genuinely hard (right-skewed, whale-dominated), so an honest R² around 0.4 is the lesson.

In [ ]:
Xr, yr, _ = get_regression_xy("total_revenue_30d")
reg_res = train_regressor(Xr.to_numpy(), yr.to_numpy(), "Random Forest")
reg_metrics = regression_metrics(reg_res.y_test, reg_res.y_pred)
print("Revenue regressor metrics:")
for k, v in reg_metrics.items():
    print(f"  {k:5s} {v:.3f}")

In [ ]:
# Actual vs predicted (the diagonal is perfect prediction).
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(reg_res.y_test, reg_res.y_pred, alpha=0.3, s=12, color="#7aa2f7")
lo, hi = float(np.min(reg_res.y_test)), float(np.max(reg_res.y_test))
ax.plot([lo, hi], [lo, hi], "--", color="#f7768e")
ax.set_xlabel("actual revenue (30d)")
ax.set_ylabel("predicted")
ax.set_title("Actual vs predicted revenue")
fig.tight_layout()
plt.show()

## 6. User clustering (K-means + PCA)

No target this time. K-means segments users by behavior; PCA projects to 2D so we can see
the segments, with silhouette as the quality score.

In [ ]:
cl = run_kmeans_pca(Xc, k=4)
print(f"silhouette: {cl.silhouette:.3f}")
print(f"PCA variance shown by 2D map: {cl.explained_variance.sum():.0%}")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
palette = ["#7aa2f7", "#f7768e", "#9ece6a", "#e0af68", "#bb9af7"]
for c in np.unique(cl.labels):
    pts = cl.coords_2d[cl.labels == c]
    ax.scatter(pts[:, 0], pts[:, 1], s=12, alpha=0.5,
               color=palette[c % len(palette)], label=f"cluster {c}")
ax.scatter(cl.centers_2d[:, 0], cl.centers_2d[:, 1], marker="X", s=160,
           color="black", zorder=5, label="centroids")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("User segments (PCA 2D)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Who are these segments? Mean feature value per cluster.
profile = cluster_profile(Xc, cl.labels)
profile

## 7. PM recommendation memo

The payoff: analysis becomes a decision. The cell below pulls the real numbers computed
above into a short memo.

In [ ]:
try:
    from IPython.display import Markdown
except ModuleNotFoundError:  # plain-Python fallback so the cell never hard-fails
    Markdown = print

auc = clf_metrics.get("roc_auc", float("nan"))
top3 = imp.head(3)["feature"].tolist()
spender_cluster = int(profile["revenue_7d"].idxmax())
spender_n = int(profile.loc[spender_cluster, "n_users"])

memo = f"""
### Memo: first-week levers for retention & revenue

**1. We can predict D7 retention well enough to act.** Random Forest reaches
**ROC-AUC = {auc:.3f}** on held-out users — well above a coin flip.

**2. The top drivers are early engagement quality:** **{top3[0]}**, **{top3[1]}**, **{top3[2]}**
— all things onboarding can influence.

**3. Revenue is predictable but hard** (R² = {reg_metrics['r2']:.2f}, MAE ≈ ${reg_metrics['mae']:.2f}).
Good for *ranking* likely spenders, not exact forecasts.

**4. There is a clear spender segment:** cluster **{spender_cluster}** ({spender_n:,} users) has the
highest early revenue — target it with premium offers.

**Recommendation:** push tutorial completion & first-session quality in onboarding; target the
spender cluster with premium offers; ship the classifier as a daily at-risk list and monitor for drift.
"""
Markdown(memo)